In [ ]:
pip install vietocr -qqq

In [ ]:
pip install torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2 --index-url https://download.pytorch.org/whl/cu118 -qqq

In [1]:
import os
import torch
from vietocr.tool.config import Cfg
from vietocr.model.trainer import Trainer
 
config = Cfg.load_config_from_name('vgg_transformer')

# Update configuration as per your requirements
dataset_params = {
    'name': 'pam_data_set_5',
    'data_root': r'C:\Users\PAM\Desktop\PPGenLabel\output\rec',
    'train_annotation': 'train.txt',
    'valid_annotation': 'val.txt',
}

params = {
    'print_every': 100,
    'valid_every': 2000,
    'iters': 10000,
    'export': '/output/ppgenlabel_transformerocr.pth',
    'metrics': 10000,
    'batch_size': 12,
}
config['dataloader']['num_workers'] = 0
config['trainer'].update(params)
config['dataset'].update(dataset_params)
config['device'] = 'cuda:0'

# Update vocabulary to include missing characters
vocab = ('aAàÀảẢãÃáÁạẠăĂằẰẳẲẵẴắẮặẶâÂầẦẩẨẫẪấẤậẬbBcCdDđĐeEèÈẻẺẽẼéÉẹẸêÊềỀểỂễỄếẾệỆ'
        'fFgGhHiIìÌỉỈĩĨíÍịỊjJkKlLmMnNoOòÒỏỎõÕóÓọỌôÔồỒổỔỗỖốỐộỘơƠờỜởỞỡỠớỚợỢpPqQrRsSt'
        'TuUùÙủỦũŨúÚụỤưƯừỪửỬữỮứỨựỰvVwWxXyYỳỲỷỶỹỸýÝỵỴzZ0123456789!"#$%&\'()*+,-./:;<'
        '=>?@[\\]^_`{|}~—…\u200ḅ̀́” “')
config['vocab'] = vocab
   
print("Starting training...")


Starting training...


In [2]:
trainer = Trainer(config, pretrained=False)
trainer.config.save('config.yml')
trainer.train()

c:\Users\PAM\.conda\envs\vietocr_312\Lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
Create train_pam_data_set_5: 100%|████████████████████████████| 1417/1417 [00:00<00:00, 1970.23it/s]

Created dataset with 1416 samples



Create valid_pam_data_set_5: 100%|██████████████████████████████| 158/158 [00:00<00:00, 2272.98it/s]

Created dataset with 157 samples



valid_pam_data_set_5 build cluster: 100%|█████████████████████| 157/157 [00:00<00:00, 156936.54it/s]


iter: 000100 - train loss: 3.529 - lr: 1.91e-05 - load time: 1.38 - gpu time: 8.37
iter: 000200 - train loss: 3.116 - lr: 3.96e-05 - load time: 1.35 - gpu time: 7.89


KeyboardInterrupt: 

In [ ]:
trainer.visualize_prediction()

In [ ]:
from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg
from PIL import Image
from vietocr.loader.prediction_loader import get_prediction_dataloader


config = Cfg.load_config_from_file(r"C:\Users\PAM\Downloads\vietocr\config.yml")

config['weights'] = r"C:\Users\PAM\Downloads\vietocr\subtitle_rec_transformerocr.pth" # <-- THAY ĐỔI ĐƯỜNG DẪN NÀY

config['device'] = 'cuda:0' # Sử dụng GPU

detector = Predictor(config)

image_folder = r'E:\raw_dataset\Total_DS\test' # <-- THAY ĐỔI ĐƯỜNG DẪN THƯ MỤC

# Tạo DataLoader
dataloader = get_prediction_dataloader(image_paths, config, batch_size=32)

# Thực hiện predict
sents, paths = detector.predict_dataloader(dataloader)

# In kết quả
for path, sent in zip(paths, sents):
    print(f'{path}: {sent}')

In [7]:
import os
import json
from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg
from PIL import Image
from vietocr.loader.prediction_loader import get_prediction_dataloader

def get_image_paths_from_dir(dir_path):
    """Lấy tất cả đường dẫn file ảnh từ một thư mục."""
    image_paths = []
    supported_formats = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.gif')
    for root, _, files in os.walk(dir_path):
        for file in files:
            if file.lower().endswith(supported_formats):
                image_paths.append(os.path.join(root, file))
    return image_paths

# 1. Load config
config = Cfg.load_config_from_file(r"C:\Users\PAM\Downloads\vietocr\config.yml")
config['weights'] = r"C:\Users\PAM\Downloads\vietocr\subtitle_rec_transformerocr.pth"
config['device'] = 'cuda:0' # Hoặc 'cpu'
output_json_path = 'results.json'

# 2. Khởi tạo predictor
detector = Predictor(config)

# 3. Lấy đường dẫn ảnh từ thư mục
image_folder = r'E:\raw_dataset\Total_DS\test'  # <-- THAY ĐỔI ĐƯỜNG DẪN THƯ MỤC CỦA BẠN
image_paths = get_image_paths_from_dir(image_folder) # <-- BƯỚC BỊ THIẾU

if not image_paths:
    print(f"Không tìm thấy ảnh nào trong thư mục: {image_folder}")
else:
    print(f"Tìm thấy {len(image_paths)} ảnh, bắt đầu dự đoán...")
    
    # 4. Tạo DataLoader
    dataloader = get_prediction_dataloader(image_paths, config, batch_size=32)
    
    # 5. Thực hiện predict
    sents, paths = detector.predict_dataloader(dataloader)
    
    results = []
    for path, sent in zip(paths, sents):
        results.append({
            'filename': os.path.basename(path), # Lấy tên file
            'path': path,                      # Giữ lại đường dẫn đầy đủ
            'prediction': sent                 # Kết quả nhận dạng
        })
        
    # Ghi danh sách kết quả vào file JSON
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=4)
        
    print(f"Đã hoàn tất! Kết quả đã được lưu vào file: {output_json_path}")

Tìm thấy 20 ảnh, bắt đầu dự đoán...
Đã hoàn tất! Kết quả đã được lưu vào file: results.json


In [3]:
from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg

# 1. Load config
config = Cfg.load_config_from_file(r"C:\Users\PAM\Downloads\vietocr\config.yml")
config['weights'] = r"C:\Users\PAM\Downloads\vietocr\subtitle_rec_transformerocr.pth"
config['device'] = 'cuda:0'
config['dataloader']['num_workers'] = 0

# 2. Khởi tạo Predictor
detector = Predictor(config)

# --- CÁCH 1: Lấy kết quả về để xử lý tiếp ---
# Chỉ cần truyền vào đường dẫn thư mục, hàm sẽ trả về một list
print("Bắt đầu dự đoán và lấy kết quả về dạng list...")
results_list = detector.predict_folder(
    folder_path=r'E:\raw_dataset\Total_DS\rec\crop_img',
    batch_size=64
)

# Bây giờ bạn có thể làm bất cứ gì với 'results_list'
print(f"Đã nhận dạng xong {len(results_list)} ảnh.")
for res in results_list[:5]: # In 5 kết quả đầu tiên
    print(f"Path: {res['path']}")
    print(f"  -> Prediction: {res['prediction']} (Confidence: {res['confidence']:.4f})\n")



Bắt đầu dự đoán và lấy kết quả về dạng list...
Tìm thấy 90815 ảnh. Bắt đầu dự đoán với batch size = 64...


KeyboardInterrupt: 